In [1]:
# hivis scraper
# takes a spreadsheet that was generated by usgs_data_scraper_1 and a blueprint image url and interval type
# appends the correct image urls where possible

# TODO:
# [x] read out datasheet datetimes and print them out seperately
# [x] guess the seconds and try to get 200 code > append url in the right row in new column
# [x] second guesser can be more biased (most seconds are the same)

import requests
from pathlib import Path
import pandas as pd

# datasheet
datasheet = 'usgs_data_01462000.xlsx'
# blue print url
image_url_blueprint = "https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___{Y}-{M}-{D}T{h}-{m}-{s}Z.jpg"
# interval type (type in the forms of minutes thhat are allowed: 0 or 0,30 or 0,15,30,45 or ...)
interval = [0]

# Load Excel file
df = pd.read_excel(datasheet)
datetime_col = "dateTime"
url_column = "image_url"
df[datetime_col] = pd.to_datetime(df[datetime_col])

second_weights = [0] * 60

def sort_numbers_by_weight():
    return sorted(range(60), key=lambda x: second_weights[x], reverse=True)

### test
success = 0
failure = 0
second_guesses = [0]*60

def append_hivis_url(dt):
    global success
    global failure
    global second_guesses
    global second_weights
    
    if dt.minute not in interval:
        return
        
    for seconds in sort_numbers_by_weight():
        print(seconds)
        request_url = image_url_blueprint.format(
            Y=dt.year, 
            M=f"{dt.month:02d}", 
            D=f"{dt.day:02d}", 
            h=f"{dt.hour:02d}", 
            m=f"{dt.minute:02d}", 
            s=f"{seconds:02d}"
        )
    
        try:
            # Send the HTTP GET request
            response = requests.get(request_url, stream=True)
            
            # Check response status
            #print(f"📡 Server responded with status code: {response.status_code}")
            
            if response.status_code == 200:
                print(f"got 200 for {request_url}")
                success += 1
                second_guesses[seconds] += 1
                second_weights[seconds] += 1
                return request_url
            else:
                failure += 1
            
        except requests.exceptions.RequestException as e:
            print("⚠️ An error occurred:")
            print(e)
        
df[url_column] = df[datetime_col].apply(append_hivis_url)
df = df[df[url_column].notna()]

df.to_excel(datasheet, index=False)

print('updated sheet')

0
1
2
3
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___2025-12-31T01-00-03Z.jpg
3
0
1
2
4
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___2025-12-31T02-00-04Z.jpg
3
4
0
1
2
5
6
7
8
9
10
11
12
13
14
15
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___2025-12-31T03-00-15Z.jpg
3
4
15
0
1
2
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___2025-12-31T04-00-02Z.jpg
2
3
4
15
0
1
5
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_Lambertville_NJ___2025-12-31T05-00-05Z.jpg
2
3
got 200 for https://usgs-nims-images.s3.amazonaws.com/720/NJ_Delaware_River_at_Lambertville_NJ/NJ_Delaware_River_at_L